## For positive_heart_CAD controls (only vcf) of 80K NGN2 derived neurons
- Example (investigate where the variant is positioned 0-based (n/2 or n/2 -1))
  - according to Max variants are centered in the middle (in the workflow documentation the controls are not mentioned ([here](https://github.com/kircherlab/MPRA_design/tree/main) and I cannot find code about controls in the workflow))
  - >C_positive_heart_CAD:ALT_rs702485_rs702485 (variant at 1-based 136th position => variant 0-based at n/2: 135)
AGGACCGGATCAACTTGGGTGGGCCCTGGATTCCCGCACTTCTGGAGCAGTCCTCAGACAGCCAAGGGATCCATCCACGGGCCAGGGCTTCCCGAGGCTGTCTCCACGGTCGCTGGGTCTCAGGAGTCGTCCTATCACCTGAGCGTGCTCGCTACTTCTGCTACCATTATGGCCACAATGACTTCCCATAAACTTAAGTCATTGAGACCATGGAATTCTGTTCCCATCCGATTCCTGTTGATGGACATTCGCTGTTTTGGCGTCATGGGAACTCCTCGGATGGTACATTGCGTGAACCGA
  - >C_positive_heart_CAD:REF_rs702485
AGGACCGGATCAACTTGGGTGGGCCCTGGATTCCCGCACTTCTGGAGCAGTCCTCAGACAGCCAAGGGATCCATCCACGGGCCAGGGCTTCCCGAGGCTGTCTCCACGGTCGCTGGGTCTCAGGAGTCGTCCTATCACCTGAGCGTGCTCACTACTTCTGCTACCATTATGGCCACAATGACTTCCCATAAACTTAAGTCATTGAGACCATGGAATTCTGTTCCCATCCGATTCCTGTTGATGGACATTCGCTGTTTTGGCGTCATGGGAACTCCTCGGATGGTACATTGCGTGAACCGA
- After looking the first rsid (rs17114036) up in dbsnp ([here](https://www.ncbi.nlm.nih.gov/snp/?term=rs17114036)) the position of the variant is 1-based NC_000001.11:56497148:A:G is the cannonical SPDI
- Process:
  - read vcf file 
  - add start: 56497149-136, end: 56497149 + 134: 56497283 
  - see here -136 leads to an additional char upfront in ucsc genome browser (1-based) => the number is 0-based [ucsc genome browser](https://genome.ucsc.edu/cgi-bin/hgTracks?db=hg38&lastVirtModeType=default&lastVirtModeExtraState=&virtModeType=default&virtMode=0&nonVirtPosition=&position=chr1%3A56497013%2D56497149&hgsid=2386769281_CRobcaBsHprD42agx8lhZWIuIlDA)

In [25]:
from importlib import reload
import pandas as pd
import sys
import os
sys.path.append('../helpful_functions')
import helpful_functions as hf
reload(hf)


import yaml

# config
config_path = "/home/kisa/coding/80K_MPRA/80K-Analysis/07_quality_control/notebooks/control_metadata/config_file.yaml"
with open(config_path) as conf:
    config = yaml.load(conf, Loader=yaml.FullLoader)
    conf.close()

In [26]:
# column names
col_name = 'name'
col_header = 'name'
col_sequence = 'sequence'
col_category = 'category'
col_class = 'class'
col_source = 'source'
col_ref = 'ref'
col_chr = 'chr'
col_start = 'start'
col_end = 'end'
col_strand = 'strand'
col_variant_class = 'variant_class'
col_variant_pos = 'variant_pos'
col_SPDI = 'SPDI'
col_allele = 'allele'
col_info = 'info'
my_col_ref_base = 'tmp_ref_base'
my_col_alt_base = 'tmp_alt_base'


interesting_columns = [col_name, col_sequence, col_category, col_class, col_source, col_ref,
                       col_chr, col_start, col_end, col_strand, col_variant_class, col_variant_pos, col_SPDI, col_allele, col_info]

In [27]:
input_fasta = config['design_file']
pre_metadata_df = hf.fasta_to_dataframe(input_fasta, columns=[col_name, col_sequence])
# split the metadata file headers by '#'
# Apply the function to each row and concatenate the results
pre_metadata_df_split = pd.concat(pre_metadata_df.apply(lambda row: hf.split_ids(row, id_col=col_name, separator='#'), axis=1).values)

# Reset the index
pre_metadata_df_split.reset_index(drop=True, inplace=True)

pre_metadata_df = pre_metadata_df_split.copy()
print(pre_metadata_df.shape[0]) # 80804

80803


In [28]:
pre_metadata_df.to_csv(config['duplicated_design_file'], sep="\t", index=False)

In [29]:
pattern = 'C_positive_heart_CAD:REF_rs12721051'
pattern = 'C_positive_heart_CAD:ALT_rs12721051_rs12721051'
pre_metadata_df.loc[pre_metadata_df[col_name] == pattern]

,name,sequence
74292,C_positive_heart_CAD:ALT_rs12721051_rs12721051,AGGACCGGATCAACTTCCTGCCTCAGCCTCATGAGTACTTGGAACT...


In [30]:
pre_metadata_df['tmp_label'] = pre_metadata_df[col_name].apply(lambda x: hf.get_label(x))

# # filter for group to get an overview
pre_metadata_df_group = pre_metadata_df.loc[pre_metadata_df['tmp_label'] == 'C_positive_heart_CAD'].copy()
pre_metadata_df_group # 99

# variant map
variant_map_path = '/home/kisa/coding/80K_MPRA/design_data/design_info/renamed_variant_region_map_unique.tsv.gz'
variant_map_path = config['variant_region_map']
variant_map = pd.read_csv(variant_map_path, sep="\t")
variant_map['tmp_label'] = variant_map['ID'].apply(hf.get_label)
variant_map_filtered = variant_map.loc[variant_map['tmp_label'] == 'C_positive_heart_CAD'].copy()
variant_map_filtered.shape[0] # 49

49

In [31]:
ref_header = variant_map_filtered['REF'].to_list()
alt_header = variant_map_filtered['ALT'].to_list()

In [32]:
pre_metadata_df_group[pre_metadata_df_group['name'].isin(ref_header + alt_header)].shape[0] # 96 (1 is missing: C_positive_heart_CAD:ALT_rs67180937_rs67180937)
pre_metadata_df_group[~pre_metadata_df_group['name'].isin(ref_header + alt_header)]

,name,sequence,tmp_label
74882,C_positive_heart_CAD:ALT_rs67180937_rs67180937,AGGACCGGATCAACTACCTTATTATTTTTTTCTTTTTTCAGTCACG...,C_positive_heart_CAD


In [33]:
# read vcf file in
vcf_file_path = '/data/cephfs-2/unmirrored/groups/kircher/MPRA/IGVF_Y1_design/design/MPRA_design/results_5k/controls/positive_heart_53_CAD.vcf'
vcf_file_path = config['positive_heart_CAD']

vcf_file = pd.read_csv(vcf_file_path, comment="#", sep="\t", header=None)
vcf_file.columns = ['chr', 'var_pos', 'rsid', 'vcf_ref', 'vcf_alt', 'strand', 'filter', 'score']

In [34]:
vcf_file["is_indel"] = vcf_file.apply(lambda row: len(row["vcf_ref"]) > 1 or len(row["vcf_alt"]) > 1, axis=1)
vcf_file.head()

,chr,var_pos,rsid,vcf_ref,vcf_alt,strand,filter,score,is_indel
0,chr1,56497149,rs17114036,A,G,.,PASS,.,False
1,chr1,56506681,rs72664324,G,A,.,PASS,.,False
2,chr1,109274968,rs12740374,G,T,.,PASS,.,False
3,chr1,156478417,rs4450010,T,G,.,PASS,.,False
4,chr1,201917641,rs34091558,T,TA,.,PASS,.,True


#### Process: 
0. Split the headers by '#' (group with duplicates)
- check the number of sequences in their duplicates
0. use variant_map to identify the headers which are within the variant map
1. identify rsid from header of the header (regex: 'rs\d{1,9}\b')
2. add chr
3. add variant pos 0-based (135)
4. compute start, end 
5. compute spdi (function)

In [35]:
# dict of chr number to refseq chromosome number
chrom_2_refseq = {"chr1": "NC_000001.11",
    "chr2": "NC_000002.12",
    "chr3": "NC_000003.12",
    "chr4": "NC_000004.12",
    "chr5": "NC_000005.10",
    "chr6": "NC_000006.12",
    "chr7": "NC_000007.14",
    "chr8": "NC_000008.11",
    "chr9": "NC_000009.12",
    "chr10": "NC_000010.11",
    "chr11": "NC_000011.10",
    "chr12": "NC_000012.12",
    "chr13": "NC_000013.11",
    "chr14": "NC_000014.9",
    "chr15": "NC_000015.10",
    "chr16": "NC_000016.10",
    "chr17": "NC_000017.11",
    "chr18": "NC_000018.10",
    "chr19": "NC_000019.10",
    "chr20": "NC_000020.11",
    "chr21": "NC_000021.9",
    "chr22": "NC_000022.11",
    "chrX": "NC_000023.11",
    "chrY": "NC_000024.10"}


def create_speedy_chromosomes(user_string, seperator="-", indices=[0,1,2,3]):
    """
    Returns SPDI identifier for given variant
    spdi: refseq_chromosome:pos:ref:alt
    @params: indices: list of indices for chromosome, position, ref and alt in user_string
    """
    if len(user_string.split(seperator)) < 4:
        raise ValueError('Not enough indices given: Expected chrom, pos, ref, alt position in user_string')
    split_string = user_string.split(seperator)
    zero_based_position = int(split_string[indices[1]]) - 1 # (input: 1-based => 0-based)
    return f'{chrom_2_refseq[split_string[indices[0]]]}:{zero_based_position}:{split_string[indices[2]]}:{split_string[indices[3]]}'


def get_spdi(row, header_col='name'):
    """
    Returns the SPDI identifier for the given variant (tested only)
    Assumption: allele need to be set beforehands
    """
    # TODO: add case for controls
    if not (hf.is_alternative(row[col_allele])):
        row[col_SPDI] = 'NA'
        return row
    # identify the variant chrom-pos-ref-alt pattern
    chrom_pos_ref_alt = hf.get_chrom_pos_ref_alt_pattern(row[header_col])
    if chrom_pos_ref_alt == "NA":
        raise ValueError('Variant pattern could not be found')
    # create the SPDI identifier
    row['SPDI'] = create_speedy_chromosomes(chrom_pos_ref_alt, seperator='-', indices=[0,1,2,3])
    return row


In [36]:
pre_metadata_df_group.shape[0] # 99

99

In [37]:
pre_metadata_df_group[pre_metadata_df_group[col_name].str.contains('#')]

,name,sequence,tmp_label


In [38]:
# Function to extract rsid
def extract_rsid(s):
    import re
    match = re.findall(r"rs\d+", s)
    return match[0] if match else None


# remove adapter from sequence (15bp of start and end):
pre_metadata_df_group[col_sequence] = pre_metadata_df_group[col_sequence].apply(lambda x: x[15:-15])
pre_metadata_df_group[col_class] = 'variant negative control'
pre_metadata_df_group[col_ref] = 'GRCh38'

# add category
pre_metadata_df_group[col_category] = 'variant'

pre_metadata_df_group['rsid'] = pre_metadata_df_group['name'].apply(extract_rsid)

# add genomic coordinates (chr, start, end, strand)
pre_metadata_df_group_merged = pre_metadata_df_group.merge(vcf_file, on='rsid', how='inner') # 97

pre_metadata_df_group_merged[col_start] = pre_metadata_df_group_merged['var_pos'] - 136 # 0-based
pre_metadata_df_group_merged[col_end] = pre_metadata_df_group_merged['var_pos'] + 134 # 1-based end

# add allele and variant pos
pre_metadata_df_group_merged[col_allele] = pre_metadata_df_group_merged[col_name].apply(lambda name: ['alt'] if 'ALT_' in name else ['ref'])
pre_metadata_df_group_merged[col_variant_pos] = pre_metadata_df_group_merged[col_name].apply(lambda name: 136 if 'ALT_' in name else 'NA')

# add variant_class
pre_metadata_df_group_merged[col_variant_class] = pre_metadata_df_group_merged['is_indel'].apply(lambda indel: 'indel' if indel else 'SNV')

# add ref, source and info

pre_metadata_df_group_merged[col_source] = 'general controls IGVF year 1 design 2023'
pre_metadata_df_group_merged[col_info] = ''

In [39]:
# add SPDI for alt
pre_metadata_df_group_merged['SPDI'] = pre_metadata_df_group_merged.apply(lambda row: create_speedy_chromosomes(f"{row['chr']}-{row['var_pos']}-{row['vcf_ref']}-{row['vcf_alt']}") if hf.is_alternative(row[col_allele]) else 'NA', axis=1)

# add SPDI for ref
# make dict for REF: [list of ALT_IDs associated to this REF] from the variant_map_filtered
ref_alt_dict = variant_map_filtered.groupby('REF')['ALT'].apply(list).to_dict()
ref_alt_dict

# # make dict for ALT_ID to SPDI from the metadata table
alt_spdi_dict = pre_metadata_df_group_merged.loc[pre_metadata_df_group_merged[col_allele].apply(hf.is_alternative)][[col_name, col_SPDI]].set_index(col_name).to_dict()[col_SPDI]
alt_spdi_dict

# # make dict for ALT_ID to variant_pos from the metadata table
alt_variant_pos_dict = pre_metadata_df_group_merged.loc[pre_metadata_df_group_merged[col_allele].apply(hf.is_alternative)][[col_name, col_variant_pos]].set_index(col_name).to_dict()[col_variant_pos]
alt_variant_pos_dict

# # make dict for ALT_ID to variant_class from the metadata table
alt_variant_class_dict = pre_metadata_df_group_merged.loc[pre_metadata_df_group_merged[col_allele].apply(hf.is_alternative)][[col_name, col_variant_class]].set_index(col_name).to_dict()[col_variant_class]
alt_variant_class_dict

# function to add a list of SPDI values from the REF to the metadata table
def add_spdi_values_2_reference(row, ref_alt_dict=ref_alt_dict, alt_spdi_dict=alt_spdi_dict, alt_variant_pos_dict=alt_variant_pos_dict, alt_variant_class_dict=alt_variant_class_dict):
    if hf.is_reference(row[col_allele]):
        # check if row[col_name] is in ref_alt_dict
        if not row[col_name] in ref_alt_dict:
            # raise exception
            raise ValueError('Reference ID not found in ref_alt_dict')
        row[col_SPDI] = [alt_spdi_dict[alt_id] for alt_id in ref_alt_dict[row[col_name]]]
        # NOTE: within variant_pos: for reference sequences a array of variant positions need to be added
        row[col_variant_pos] = [int(alt_variant_pos_dict[alt_id]) for alt_id in ref_alt_dict[row[col_name]]]
        row[col_variant_class] = [alt_variant_class_dict[alt_id] for alt_id in ref_alt_dict[row[col_name]]]
        row[col_allele] = ['ref' for _ in ref_alt_dict[row[col_name]]]
    return row

pre_metadata_df_group_merged = pre_metadata_df_group_merged.apply(lambda row: add_spdi_values_2_reference(row, ref_alt_dict=ref_alt_dict, alt_spdi_dict=alt_spdi_dict, alt_variant_pos_dict=alt_variant_pos_dict, alt_variant_class_dict=alt_variant_class_dict), axis=1)

In [40]:
# function to generate arrays out off the columns
def make_column_arrays(row):
    """
    create arrays for the required columns
    """
    allele = row[col_allele]
    SPDI = row[col_SPDI]
    variant_pos = row[col_variant_pos]
    variant_class = row[col_variant_class]

    if allele == 'ref' or allele == 'alt': # only "alt" is string
        row[col_allele] = [allele]
    if isinstance(SPDI, str): # only for alt sequences this is true
        if SPDI != "NA":
            row[col_SPDI] = [SPDI]
    if isinstance(variant_pos, float):
        row[col_variant_pos] = [int(variant_pos)]
    elif isinstance(variant_pos, int):
        row[col_variant_pos] = [int(variant_pos)]
    if isinstance(variant_class, str):
        if row[col_variant_class] in ['SNV', 'indel']:
                row[col_variant_class] = [variant_class]
    if not isinstance(row[col_class], str):
        print(row[col_name])
    return row

In [41]:
pre_metadata_df_group_merged = pre_metadata_df_group_merged.apply(make_column_arrays, axis=1)

In [42]:
pre_metadata_df_group_merged.columns

Index(['name', 'sequence', 'tmp_label', 'class', 'ref', 'category', 'rsid',
       'chr', 'var_pos', 'vcf_ref', 'vcf_alt', 'strand', 'filter', 'score',
       'is_indel', 'start', 'end', 'allele', 'variant_pos', 'variant_class',
       'source', 'info', 'SPDI'],
      dtype='object')

In [43]:
# check if the row sum is equal to the initial row number
print(f"Expected number of rows: {pre_metadata_df.loc[pre_metadata_df['tmp_label'] == 'C_positive_heart_CAD'].shape[0]}")
print(f'Number of rows with matching coordinsates: {pre_metadata_df_group_merged.shape[0]}')

Expected number of rows: 99
Number of rows with matching coordinsates: 99


In [44]:
group_name = 'C_positive_heart_CAD'
output_dir = config['final_output_dir']
output_path = os.path.join(output_dir, group_name)
# Write DataFrame to TSV file
pre_metadata_df_group_merged[interesting_columns].to_csv(os.path.join(output_path, f'{group_name}.metadata.tmp.tsv.gz'), sep='\t', index=False, na_rep='NA', compression='gzip')
os.system(f'zcat {output_path}/{group_name}.metadata.tmp.tsv.gz | sed "s/\'/\\"/g" | gzip -c > {output_path}/{group_name}.metadata.tsv.gz')

0